In [0]:

BEGIN
  DECLARE v_month_start DATE;
  DECLARE v_end         DATE;

  SET v_month_start = (
    SELECT add_months(MAX(report_month), 1)
    FROM data_exploration_hub.marketing_hub.mkt_dashboard_concessions_monthly
  );
  SET v_end = date_sub(current_date(), 1);

  WHILE last_day(v_month_start) <= v_end DO
    DELETE FROM data_exploration_hub.marketing_hub.mkt_dashboard_concessions_monthly
    WHERE report_month = v_month_start;

    INSERT INTO data_exploration_hub.marketing_hub.mkt_dashboard_concessions_monthly

    WITH params AS (
        SELECT
            v_month_start                               AS month_start,
            LAST_DAY(v_month_start)                     AS month_end
    ),
    -- ---------------------------------------------------------------------
    -- Active CC members as of month end (reuses bal_all_active_union pattern
    -- via mkt_subscriber_profile_snapshot, same as Table 2 / Table 4)
    -- ---------------------------------------------------------------------
    active_cc_members AS (
        SELECT DISTINCT s.cde_id
        FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot s
        CROSS JOIN params p
        WHERE s.snapshot_date_id = CAST(DATE_FORMAT(p.month_end, 'yyyyMMdd') AS INT)
        AND s.renewal_status IN ('Active', 'Active_Non_Renewing', 'Active_Annual_Upgrade')
    ),

    -- ---------------------------------------------------------------------
    -- Item classification (cpx_d_item), one row per item with category tag
    -- ---------------------------------------------------------------------
    item_categories AS (
        SELECT
            itemskey,
            CASE
                WHEN (
                        Item_Description LIKE '%POPCORN%'
                        OR Item_Description LIKE '%CORN%UPSIZE%'
                        OR Item_Description LIKE '%UPSIZE%CORN%'
                    )
                    AND Item_Description NOT LIKE '%CORN DOG%'
                    THEN 'popcorn'
                WHEN Item_Description LIKE '%DRINK%CUP%'
                    OR Item_Description LIKE '%VIP%DRINK%'
                    OR Item_Description LIKE '%UPSIZE%FOUNTAIN%'
                    THEN 'fountain_drinks'
                WHEN Item_Class_Description1 IN ('VHO-CANDY', 'CANDY', 'SMALL CANDY')
                    AND Item_Description NOT LIKE 'DELIVERY%'
                    AND Item_Description NOT LIKE 'VHO DELIVERY%'
                    AND Item_Description NOT LIKE 'SNACK%'
                    THEN 'candy'
                WHEN Item_Description LIKE '%NACHO%'
                    OR Item_Description LIKE '%HOT DOG%'
                    THEN 'hot_foods'
                WHEN Item_Class_Description1 IN (
                        'VHO-MERCH', 'Retired Merch Codes', 'TRR - MERCH STORE',
                        'TRR - MERCHANDISING - G&E', 'VHO-GROUP - MERCHANDISE',
                        'VHO-STARBUCKS MERCHANDISE'
                    )
                    THEN 'merch'
                ELSE NULL
            END AS concession_category
        FROM cpx_dataanalytics_gold.customer360.cpx_d_item
    ),

    -- ---------------------------------------------------------------------
    -- EXH concession transactions (Cineplex Theatre only), tagged by category
    -- and population (cc / scene / total_biz determined at query time below)
    -- ---------------------------------------------------------------------
    exh_concession_lines AS (
        SELECT
            t.CDE_ID,
            t.Transaction_Number,
            t.Net_Revenue,
            t.Quantity,
            ic.concession_category,
            CASE WHEN t.CineClubFLAG = 'Yes' THEN 1 ELSE 0 END AS is_cc,
            CASE WHEN t.SCENEFLAG = 'Yes' AND t.CineClubFLAG IS DISTINCT FROM 'Yes' THEN 1 ELSE 0 END AS is_scene
        FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
        JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc
            ON t.LocationId = loc.LocationID
        JOIN item_categories ic
            ON t.ItemSkey = ic.itemskey
        CROSS JOIN params p
        WHERE loc.LC_Location_Type_Description = 'Cineplex Theatre'
        AND t.ConcessionFLAG = 'Yes'
        AND ic.concession_category IS NOT NULL
        AND t.VisitDateId BETWEEN CAST(DATE_FORMAT(p.month_start, 'yyyyMMdd') AS INT)
                                AND CAST(DATE_FORMAT(p.month_end, 'yyyyMMdd') AS INT)
    ),

    -- ---------------------------------------------------------------------
    -- LBE gaming / F&B transactions (Rec Room / Playdium), tagged by population
    -- ---------------------------------------------------------------------
    lbe_lines AS (
        SELECT
            t.CDE_ID,
            t.Transaction_Number,
            t.Net_Revenue,
            CASE WHEN i.Item_Class_Description1 IN (
                    'TRR - AMUSEMENT - G&E','TRR - AMUSEMENT - GAMES',
                    'TRR - AMUSEMENT - PHYSICAL GAMES','TRR - PROMO - AMUSEMENT'
                ) THEN 'gaming'
                WHEN i.Item_Class_Description1 IN (
                    'PDM - MENU FOOD - CANDY','TRR - INV ALC BEV - BOTTLE','TRR - INV ALC BEV - CAN',
                    'TRR - INV ALC BEV - WINE','TRR - MENU ALC BEV - COCKTAIL','TRR - MENU ALC BEV - DRAUGHT BEER',
                    'TRR - MENU BEV - NA COLD BEV','TRR - MENU BEV - NA HOT BEV','TRR - MENU FOOD - APP',
                    'TRR - MENU FOOD - DESSERT','TRR - MENU FOOD - KIDS','TRR - MENU FOOD - PIZZA',
                    'TRR - MENU FOOD - POUTINE','TRR - MENU FOOD - SANDWICHES','TRR - MOD MENU - ADD-ON',
                    'TRR - INV BEV - NA BEV','TRR - MENU ALC BEV - LIQUOR','TRR - MENU ALC BEV - WINE',
                    'TRR - MENU FOOD - ENTRÉE','TRR - MENU FOOD - SALAD','VHO-LICENSED FOOD',
                    'TRR - INV ALC BEV - LIQUOR','VHO-CANDY','VIP Food - Addons','VHO-LICENSED DRINKS',
                    'TRR - MENU FOOD - BRUNCH','VHO-DRINK'
                ) THEN 'fnb'
                ELSE NULL
            END AS lbe_category,
            CASE WHEN t.CineClubFLAG = 'Yes' THEN 1 ELSE 0 END AS is_cc,
            CASE WHEN t.SCENEFLAG = 'Yes' AND t.CineClubFLAG IS DISTINCT FROM 'Yes' THEN 1 ELSE 0 END AS is_scene
        FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
        JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc
            ON t.LocationId = loc.LocationID
        JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i
            ON t.ItemSkey = i.itemskey
        CROSS JOIN params p
        WHERE loc.LC_Location_Type_Description LIKE 'Restaurant%'
        AND loc.LC_Trade_Brand_Description IN ('Rec Room', 'Playdium')
        AND t.VisitDateId BETWEEN CAST(DATE_FORMAT(p.month_start, 'yyyyMMdd') AS INT)
                                AND CAST(DATE_FORMAT(p.month_end, 'yyyyMMdd') AS INT)
    ),

    -- ---------------------------------------------------------------------
    -- Per-population, per-category rollups (CC / Scene / Total_Biz)
    -- ---------------------------------------------------------------------
    concession_agg AS (
        SELECT population_type, concession_category,
            SUM(Net_Revenue) AS revenue,
            COUNT(DISTINCT Transaction_Number) AS transactions,
            SUM(Quantity) AS quantity,
            COUNT(DISTINCT CDE_ID) AS members
        FROM (
            SELECT 'CC' AS population_type, concession_category, Net_Revenue, Transaction_Number, Quantity, CDE_ID
            FROM exh_concession_lines WHERE is_cc = 1
            UNION ALL
            SELECT 'Scene', concession_category, Net_Revenue, Transaction_Number, Quantity, CDE_ID
            FROM exh_concession_lines WHERE is_scene = 1
            UNION ALL
            SELECT 'Total_Biz', concession_category, Net_Revenue, Transaction_Number, Quantity, CDE_ID
            FROM exh_concession_lines
        )
        GROUP BY population_type, concession_category
    ),

    lbe_agg AS (
        SELECT population_type, lbe_category,
            SUM(Net_Revenue) AS revenue,
            COUNT(DISTINCT Transaction_Number) AS transactions,
            COUNT(DISTINCT CDE_ID) AS members
        FROM (
            SELECT 'CC' AS population_type, lbe_category, Net_Revenue, Transaction_Number, CDE_ID
            FROM lbe_lines WHERE is_cc = 1
            UNION ALL
            SELECT 'Scene', lbe_category, Net_Revenue, Transaction_Number, CDE_ID
            FROM lbe_lines WHERE is_scene = 1
            UNION ALL
            SELECT 'Total_Biz', lbe_category, Net_Revenue, Transaction_Number, CDE_ID
            FROM lbe_lines
        )
        WHERE lbe_category IS NOT NULL
        GROUP BY population_type, lbe_category
    ),

    populations AS (
        SELECT explode(array('CC', 'Scene', 'Total_Biz')) AS population_type
    )

    -- ---------------------------------------------------------------------
    -- Final wide-per-row-per-population output
    -- ---------------------------------------------------------------------
    SELECT
        p.month_start                                                                   AS report_month,
        pop.population_type,

        MAX(CASE WHEN ca.concession_category = 'popcorn' THEN ca.revenue END)          AS concession_popcorn_revenue,
        MAX(CASE WHEN ca.concession_category = 'popcorn' THEN ca.transactions END)     AS concession_popcorn_transactions,
        MAX(CASE WHEN ca.concession_category = 'popcorn' THEN ca.quantity END)         AS concession_popcorn_quantity,
        MAX(CASE WHEN ca.concession_category = 'popcorn' THEN ca.members END)          AS concession_popcorn_members,

        MAX(CASE WHEN ca.concession_category = 'fountain_drinks' THEN ca.revenue END)      AS concession_fountain_drinks_revenue,
        MAX(CASE WHEN ca.concession_category = 'fountain_drinks' THEN ca.transactions END) AS concession_fountain_drinks_transactions,
        MAX(CASE WHEN ca.concession_category = 'fountain_drinks' THEN ca.quantity END)     AS concession_fountain_drinks_quantity,
        MAX(CASE WHEN ca.concession_category = 'fountain_drinks' THEN ca.members END)      AS concession_fountain_drinks_members,

        MAX(CASE WHEN ca.concession_category = 'candy' THEN ca.revenue END)            AS concession_candy_revenue,
        MAX(CASE WHEN ca.concession_category = 'candy' THEN ca.transactions END)       AS concession_candy_transactions,
        MAX(CASE WHEN ca.concession_category = 'candy' THEN ca.quantity END)           AS concession_candy_quantity,
        MAX(CASE WHEN ca.concession_category = 'candy' THEN ca.members END)            AS concession_candy_members,

        MAX(CASE WHEN ca.concession_category = 'hot_foods' THEN ca.revenue END)        AS concession_hot_foods_revenue,
        MAX(CASE WHEN ca.concession_category = 'hot_foods' THEN ca.transactions END)   AS concession_hot_foods_transactions,
        MAX(CASE WHEN ca.concession_category = 'hot_foods' THEN ca.quantity END)       AS concession_hot_foods_quantity,
        MAX(CASE WHEN ca.concession_category = 'hot_foods' THEN ca.members END)        AS concession_hot_foods_members,

        MAX(CASE WHEN ca.concession_category = 'merch' THEN ca.revenue END)            AS concession_merch_revenue,
        MAX(CASE WHEN ca.concession_category = 'merch' THEN ca.transactions END)       AS concession_merch_transactions,
        MAX(CASE WHEN ca.concession_category = 'merch' THEN ca.quantity END)           AS concession_merch_quantity,
        MAX(CASE WHEN ca.concession_category = 'merch' THEN ca.members END)            AS concession_merch_members,

        MAX(CASE WHEN la.lbe_category = 'gaming' THEN la.revenue END)                  AS gaming_revenue,
        MAX(CASE WHEN la.lbe_category = 'gaming' THEN la.transactions END)             AS gaming_transactions,
        MAX(CASE WHEN la.lbe_category = 'gaming' THEN la.members END)                  AS gaming_members,

        MAX(CASE WHEN la.lbe_category = 'fnb' THEN la.revenue END)                     AS fnb_revenue,
        MAX(CASE WHEN la.lbe_category = 'fnb' THEN la.transactions END)                AS fnb_transactions,
        MAX(CASE WHEN la.lbe_category = 'fnb' THEN la.members END)                     AS fnb_members

    FROM populations pop
    CROSS JOIN params p
    LEFT JOIN concession_agg ca ON ca.population_type = pop.population_type
    LEFT JOIN lbe_agg la ON la.population_type = pop.population_type
    GROUP BY p.month_start, pop.population_type
    ORDER BY pop.population_type;
    
    SET v_month_start = add_months(v_month_start, 1);
  END WHILE;
END;